# Thermal Hotspot Classification (PV Fault Imagery)

This notebook trains and compares multiple transfer-learning CNN backbones (DenseNet201, ResNet50, EfficientNetB0) to classify PV thermal images into:

- Clean (no hotspot)
- Hotspot (single hotspot)

MLflow is used to log metrics, plots, and trained models for fair comparison.

## Import Necessary Libraries

In [86]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.style.use("ggplot")

import cv2

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc,
    precision_recall_curve,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    log_loss,
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)
from sklearn.calibration import calibration_curve

# Tensorflow
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import DenseNet201, ResNet50, EfficientNetB0
from tensorflow.keras.metrics import Precision, Recall, AUC
from huggingface_hub import hf_hub_download
import zipfile

# MLflow
import mlflow

## Reproducability and Dataset Acquisition

A fixed random seed is used for NumPy and TensorFlow to improve experiment reproducability across runs.

Furthermore, the dataset is downloaded from Hugging Face (`Solar-PV-Clean-Hotspot-Images`), and extracted locally into a `dataset/` directory.

In [ ]:
# reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Download the zip file
zip_path = hf_hub_download(
    repo_id="seyeddd/solar_pv_single_hotspot_clean_images",
    filename="cleaned_dataset.zip",
    repo_type="dataset"
)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall("dataset")

In [88]:
# Constants used in comparisons
IMG_SIZE = (224, 224)
BATCH_SIZE = 64
EPOCHS = 20

In [89]:
# Directories for images
train_dir = "dataset/train"
valid_dir = "dataset/valid"
test_dir  = "dataset/test"

In [90]:
# Set MLflow tracking URI
LOCAL_MLFLOW_DIR = Path.home() / "mlflow_local"   # not in OneDrive
LOCAL_MLFLOW_DIR.mkdir(parents=True, exist_ok=True)

# Makes MLflow create mlruns in ~/mlflow_local where it can write normally
mlflow.set_tracking_uri(f"file://{LOCAL_MLFLOW_DIR.as_posix()}")

# MLflow experiment name
MLFLOW_EXPERIMENT = "PV_Hotspot_DL_Model_Comparison"

In [91]:
# Output root for artifacts
OUTPUT_ROOT = Path.home() / "dl_mlflow_outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

## Image Preprocessing

Each image is preprocessed before entering the network:

1. Convert RGB → Grayscale (to focus on intensity patterns)
2. Apply Gaussian Blur (reduce noise)
3. Normalize to [0, 1]
4. Expand back to 3 channels (required by ImageNet backbones)

In [92]:
def preprocess(img) -> np.ndarray:
    """
    Preprocess a single image before feeding to the CNN.

    Steps:
    1) Convert RGB -> Grayscale (reduces color noise, focus on thermal frequency)
    2) Gaussian blur (denoise)
    3) Normalize pixel values to [0,1]
    4) Expand back to 3 channels (required by ImageNet pretrained models)

    Args:
        img (np.ndarray): Input image array from ImageDataGenerator.

    Returns:
        np.ndarray: Preprocessed image with shape (H, W, 3) and float values in [0,1].
    """

    if len(img.shape) == 3:
        img_gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    else:
        img_gray = img

    img_blurred = cv2.GaussianBlur(img_gray, ksize=(3, 3), sigmaX=0)
    img_normalized = img_blurred / 255.0

    if img_normalized.ndim == 2:
        img_normalized = np.stack([img_normalized] * 3, axis=-1)

    return img_normalized

## Data Pipeline

Images are loaded using `ImageDataGenerator` with:

- Training augmentation (rotation, shear, zoom, horizontal flip)
- Validation/test with only preprocessing (no augmentation)

All images are resized to 224x224 and batched for training.

In [93]:
# Train generator: augmentation + preprocessing
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess,
    rotation_range=10,
    shear_range=0.01,
    zoom_range=0.10,
    horizontal_flip=True,
    fill_mode="nearest"
)

# Validation/test generators: preprocessing only
val_test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess
)

In [94]:
train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=True,
    seed=SEED
)

val_gen = val_test_datagen.flow_from_directory(
    valid_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

test_gen = val_test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

Found 2742 images belonging to 2 classes.
Found 649 images belonging to 2 classes.
Found 650 images belonging to 2 classes.


In [95]:
# View sample counts
print(f"Training samples: {train_gen.samples}")
print(f"Validation samples: {val_gen.samples}")
print(f"Test samples: {test_gen.samples}")
print(f"Classes: {train_gen.class_indices}")

Training samples: 2742
Validation samples: 649
Test samples: 650
Classes: {'clean': 0, 'single_hotspot': 1}


## Model Architecture (Transfer Learning)

Each backbone uses ImageNet weights with the top removed. The backbone is frozen and a small custom head is added:

- Global Average Pooling
- Dense layers + Batch Normalization
- Sigmoid output for binary classification

Loss: Binary Cross Entropy

Optimizer: Adam

Early stopping monitors validation AUC.

In [96]:
def build_transfer_model(base_model, model_name: str, lr: float = 1e-3):
    """
    Builds a transfer-learning model using a pretrained backbone.

    Steps:
        1) Freeze pretrained backbone weights (feature extractor).
        2) Add a small trained classification head.
        3) Compile with Adam + binary cross-entropy

    Args:
        base_model (tf.keras.Model): Pretrained CNN backbone without top layers.
        model_name (str): Named used for identification and MLflow runs.
        lr (float): Learning rate for Adam optimizer.

    Returns:
        tf.keras.Model: Compiled Keras model ready for training.
    """
    
    base_model.trainable = False

    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation="relu"),
        layers.BatchNormalization(),
        layers.Dense(64, activation="relu"),
        layers.BatchNormalization(),
        layers.Dense(32, activation="relu"),
        layers.BatchNormalization(),
        layers.Dense(1, activation="sigmoid")
    ], name=model_name)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            Precision(name="precision"),
            Recall(name="recall"),
            AUC(name="auc")
        ]
    )
    return model

In [97]:
def get_models(lr: float = 1e-3):
    """
    Returns ground truth labels, predicted probabilities, and threshold predictions
    for the test generator.
    """

    densenet = DenseNet201(weights="imagenet", include_top=False, input_shape=(*IMG_SIZE, 3))
    resnet = ResNet50(weights="imagenet", include_top=False, input_shape=(*IMG_SIZE, 3))
    effnet = EfficientNetB0(weights="imagenet", include_top=False, input_shape=(*IMG_SIZE, 3))

    return {
        "DenseNet201": build_transfer_model(densenet, "DenseNet201_Model", lr=lr),
        "ResNet50": build_transfer_model(resnet, "ResNet50_Model", lr=lr),
        "EfficientNetB0": build_transfer_model(effnet, "EfficientNetB0_Model", lr=lr),
    }

## Model Evaluation

Models are evaluated using:

- Accuracy, Balanced Accuracy
- Precision, Recall, F1-Score
- ROC-AUC and PR-AUC
- Log Loss
- Cohen's Kappa

To ensure standardized evaluation across models, helper functions are implemented to:

- Generate and save confusion matrices
- Plot ROC curves and compute ROC-AUC
- Plot Precision–Recall curves and compute PR-AUC
- Visualize training curves (Accuracy, Loss, AUC)

All figures are saved as artifacts for experiment tracking and reproducibility.

In [98]:
def save_confusion_matrix(y_true, y_pred, labels, out_path: Path, title: str):
    """
    Generate and save a confusion matrix plot.

    Args:
        y_true: Ground truth labels.
        y_pred: Predicted class labels.
        labels (list): Class label names for display.
        out_path (Path): File path to save the figure.
        title (str): Plot title.

    Returns:
        None
    """

    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)

    plt.figure(figsize=(6, 5))
    disp.plot(cmap=plt.cm.Reds, values_format="d")
    plt.title(title, fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()


def save_roc_curve(y_true, y_prob, out_path: Path, title: str):
    """
    Generate and save the ROC curve for binary classification.

    Args:
        y_true: Ground truth binary labels.
        y_prob: Predicted probabilities for the positive class.
        out_path (Path): File path to save the figure.
        title (str): Plot title.

    Returns:
        float: Computed ROC-AUC score.
    """

    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_score = auc(fpr, tpr)

    plt.figure(figsize=(7, 5))
    plt.plot(fpr, tpr, label=f"AUC = {roc_score:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title, fontsize=14, fontweight="bold")
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()

    return roc_score


def save_pr_curve(y_true, y_prob, out_path: Path, title: str):
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)

    plt.figure(figsize=(7, 5))
    plt.plot(recall, precision, label=f"AP = {ap:.4f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(title, fontsize=14, fontweight="bold")
    plt.legend(loc="lower left")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()

    return ap


def save_ks_chart(y_true, y_prob, out_path: Path, title: str):
    """
    Save a KS chart (TPR vs FPR across thresholds) and return KS statistic + best threshold.
    """
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    ks_values = tpr - fpr

    ks_stat = float(np.max(ks_values))
    ks_idx = int(np.argmax(ks_values))
    ks_thr = float(thresholds[ks_idx])

    plt.figure(figsize=(8, 6))
    plt.plot(thresholds, tpr, label="TPR")
    plt.plot(thresholds, fpr, label="FPR")

    # Shade KS gap
    plt.fill_between(thresholds, tpr, fpr, where=(tpr >= fpr), alpha=0.2)

    # Mark KS point
    plt.scatter([ks_thr], [tpr[ks_idx]], s=60, label=f"KS={ks_stat:.4f} @ thr={ks_thr:.4f}")
    plt.scatter([ks_thr], [fpr[ks_idx]], s=60)

    plt.title(title, fontsize=14, fontweight="bold")
    plt.xlabel("Threshold")
    plt.ylabel("Rate")
    plt.xlim([0, 1])
    plt.ylim([0, 1])
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()

    return ks_stat, ks_thr


def save_calibration_curve(y_true, y_prob, out_path: Path, title: str, n_bins: int = 10):
    """
    Save calibration curve (reliability diagram) and return Brier score.
    """
    prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=n_bins)
    brier = float(brier_score_loss(y_true, y_prob))

    plt.figure(figsize=(7, 6))
    plt.plot(prob_pred, prob_true, marker="o", label="Model")
    plt.plot([0, 1], [0, 1], linestyle="--", label="Perfectly calibrated")
    plt.title(f"{title}\nBrier={brier:.4f}", fontsize=14, fontweight="bold")
    plt.xlabel("Mean predicted probability")
    plt.ylabel("Fraction of positives")
    plt.xlim([0, 1])
    plt.ylim([0, 1])
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()

    return brier


def save_training_curves(history, out_path: Path, model_name: str):
    """
    Generate and save training curves for a deep learning model.

    The figure contains:
        - Training vs Validation Accuracy
        - Training vs Validation Loss
        - Training vs Validation AUC

    Args:
        history (tf.keras.callbacks.History): Training history object.
        out_path (Path): File path to save the figure.
        model_name (str): Name of the model for labeling plots.

    Returns:
        None
    """

    fig = plt.figure(figsize=(16, 4))

    # Accuracy
    ax1 = plt.subplot(1, 3, 1)
    ax1.plot(history.history.get("accuracy", []), label="Train")
    ax1.plot(history.history.get("val_accuracy", []), label="Val")
    ax1.set_title(f"Accuracy - {model_name}")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Accuracy")
    ax1.legend()

    # Loss
    ax2 = plt.subplot(1, 3, 2)
    ax2.plot(history.history.get("loss", []), label="Train")
    ax2.plot(history.history.get("val_loss", []), label="Val")
    ax2.set_title(f"Loss - {model_name}")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Loss")
    ax2.legend()

    # AUC
    ax3 = plt.subplot(1, 3, 3)
    if "auc" in history.history and "val_auc" in history.history:
        ax3.plot(history.history["auc"], label="Train")
        ax3.plot(history.history["val_auc"], label="Val")
        ax3.set_title(f"AUC - {model_name}")
        ax3.set_xlabel("Epoch")
        ax3.set_ylabel("AUC")
        ax3.legend()
    else:
        ax3.text(0.1, 0.5, "AUC history not found", fontsize=12)

    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close(fig)

In [99]:
def evaluate_binary(y_true, y_pred, y_prob):
    """
    Computes standard evaluation metrics for binary classification.

    Metrics computed:
        - Accuracy
        - Precision
        - Recall
        - F1-score
        - Log Loss
        - ROC-AUC
        - PR-AUC

    Args:
        y_true: Ground truth labels (0 or 1).
        y_pred: Predicted class labels (0 or 1).
        y_prob: Predicted probabilites for the positive class.

    Returns:
        dict: Dictionary containing all computed metrics.
    """

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "log_loss": log_loss(y_true, y_prob),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
    }
    return metrics

In [100]:
def get_preds(model, test_gen):
    """
    Generate predictions and probabilities from a trained binary classification model.

    Args:
        model (tf.keras.Model): Trained Keras model.
        test_gen: Test data generator.

    Returns:
        tuple:
            y_true (np.ndarray): Ground truth binary labels.
            y_prob (np.ndarray): Predicted probabilities for positive class.
            y_pred (np.ndarray): Threshold binary predictions (0 or 1).
    """

    test_gen.reset()    # Ensure predicions start from first batch
    y_prob = model.predict(test_gen, verbose=0).ravel() # Predicted probabilities
    y_pred = (y_prob > 0.5).astype(int)
    y_true = test_gen.classes.astype(int)   # true labels from generator
    return y_true, y_prob, y_pred

## Experiment Tracking (MLflow)

For each model, MLflow logs:

- Hyperparameters (image size, batch size, learning rate, augmentation settings)
- Metrics (accuracy, AUC, etc.)
- Artifacts (plots, classification report)
- Final trained model

This will ensure transparent comparison across backbones.

In [101]:
def train_one_model(model_name: str, model: tf.keras.Model, lr: float):
    """
    Trains one deep learning model and logs the full experiment to MLflow.

    Logs:
        - Training configuration (params)
        - Training curves + evaluation plots (artifacts)
        - Classification report (artifact)
        - Metrics (accuracy, AUC, PR-AUC, etc.)
        - Saved model (.keras + MLflow model)

    Returns:
        dict: Evaluation metrics computed on the test set.
    """
    
    run_out = OUTPUT_ROOT / model_name
    run_out.mkdir(parents=True, exist_ok=True)

    class_labels = list(test_gen.class_indices.keys())

    # MLflow run
    with mlflow.start_run(run_name=model_name):
        # Log params
        mlflow.log_param("model_name", model_name)
        mlflow.log_param("img_size", f"{IMG_SIZE[0]}x{IMG_SIZE[1]}")
        mlflow.log_param("batch_size", BATCH_SIZE)
        mlflow.log_param("epochs", EPOCHS)
        mlflow.log_param("seed", SEED)
        mlflow.log_param("optimizer", "Adam")
        mlflow.log_param("learning_rate", lr)

        # Aug params
        mlflow.log_param("rotation_range", 10)
        mlflow.log_param("shear_range", 0.01)
        mlflow.log_param("zoom_range", 0.10)
        mlflow.log_param("horizontal_flip", True)
        mlflow.log_param("fill_mode", "nearest")

        # Train
        earlystop = EarlyStopping(
            monitor="val_auc",
            patience=5,
            restore_best_weights=True,
            mode="max"
        )

        history = model.fit(
            train_gen,
            validation_data=val_gen,
            epochs=EPOCHS,
            callbacks=[earlystop],
            verbose=1
        )

        test_gen.reset()
        keras_results = model.evaluate(test_gen, return_dict=True, verbose=0)
        for k, v in keras_results.items():
            mlflow.log_metric(f"keras_{k}", float(v))   # keras_loss, keras_accuracy, keras_precision, keras_recall, keras_auc

        # Save + log training curves
        curves_path = run_out / "training_curves.png"
        save_training_curves(history, curves_path, model_name)
        mlflow.log_artifact(str(curves_path))

        # Evaluate
        y_true, y_prob, y_pred = get_preds(model, test_gen)

        # classification report
        report = classification_report(y_true, y_pred, target_names=class_labels)
        report_path = run_out / "classification_report.txt"
        report_path.write_text(report)
        mlflow.log_artifact(str(report_path))

        # plots
        cm_path = run_out / "confusion_matrix.png"
        roc_path = run_out / "roc_curve.png"
        pr_path = run_out / "precision_recall_curve.png"

        save_confusion_matrix(y_true, y_pred, class_labels, cm_path, f"Confusion Matrix - {model_name}")
        roc_auc_plot = save_roc_curve(y_true, y_prob, roc_path, f"ROC Curve - {model_name}")
        pr_ap_plot   = save_pr_curve(y_true, y_prob, pr_path, f"PR Curve - {model_name}")

        mlflow.log_artifact(str(cm_path))
        mlflow.log_artifact(str(roc_path))
        mlflow.log_artifact(str(pr_path))

        # metrics (sklearn)
        ks_path = run_out / "ks_chart.png"
        cal_path = run_out / "calibration_curve.png"

        ks_stat, ks_thr = save_ks_chart(y_true, y_prob, ks_path, f"KS Chart - {model_name}")
        brier = save_calibration_curve(y_true, y_prob, cal_path, f"Calibration Curve - {model_name}", n_bins=10)

        mlflow.log_artifact(str(ks_path))
        mlflow.log_artifact(str(cal_path))

        # metrics (sklearn)
        metrics = evaluate_binary(y_true, y_pred, y_prob)
        metrics["ks"] = float(ks_stat)
        metrics["ks_threshold"] = float(ks_thr)
        metrics["brier"] = float(brier)
        metrics["roc_auc_curve_auc"] = float(roc_auc_plot)
        metrics["pr_curve_ap"] = float(pr_ap_plot)

        # log metrics once
        for k, v in metrics.items():
            mlflow.log_metric(k, float(v))

        # Save metrics_summary.csv (include everything)
        metrics_df = pd.DataFrame(list(metrics.items()), columns=["metric", "value"])
        metrics_csv = run_out / "metrics_summary.csv"
        metrics_df.to_csv(metrics_csv, index=False)
        mlflow.log_artifact(str(metrics_csv))

        # Save model
        model_path = run_out / f"{model_name}.keras"
        model.save(model_path)

        # Log model to mlflow
        mlflow.tensorflow.log_model(model, artifact_path="model")

        return metrics

In [102]:
def train_all_models(lr: float = 1e-3):
    """
    Trains and evaluates multiple deep learning backbones.

    Steps:
        1) Set MLflow experiment.
        2) Initialize all transfer-learning models.
        3) Train each model individually.
        4) Log metrics and artifacts to MLflow.
        5) Aggregrate results into a comparison table.
        6) Save summary results as a CSV.

    Args:
        lr (float): Learning rate for all models (default = 1e-3).

    Returns:
        pd.Dataframe: Sorted performance summary across all models.
    """

    # Define experiment tracking group
    mlflow.set_experiment(MLFLOW_EXPERIMENT)

    # Load all backbone architectures
    models_dict = get_models(lr=lr)
    all_results = []

    # Train each model sequentially
    for name, model in models_dict.items():
        print("\n" + "-" * 90)
        print(f"Training + Logging: {name}")
        print("-" * 90)
        metrics = train_one_model(name, model, lr)
        all_results.append({"model": name, **metrics})

    # Create comparison table sorted by ROC-AUC
    results_df = pd.DataFrame(all_results).sort_values(by="roc_auc", ascending=False)

    # Save summary to disk
    results_df.to_csv(OUTPUT_ROOT / "all_models_summary.csv", index=False)
    print("\nSaved:", OUTPUT_ROOT / "all_models_summary.csv")
    print(results_df[["model", "accuracy",
                      "precision", "recall", "f1", "roc_auc",
                      "pr_auc", "log_loss"]])

    return results_df

In [103]:
# Run the pipeline
train_all_models(lr=1e-3)


------------------------------------------------------------------------------------------
Training + Logging: DenseNet201
------------------------------------------------------------------------------------------


/Users/seyedrumaiz/Library/CloudStorage/OneDrive-InformaticsInstituteofTechnology/DSGP/solar-panel-fault-mapping/.venv/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
 9/43 ━━━━━━━━━━━━━━━━━━━━ 2:28 4s/step - accuracy: 0.7730 - auc: 0.8141 - loss: 0.5147 - precision: 0.8451 - recall: 0.7758

/Users/seyedrumaiz/Library/CloudStorage/OneDrive-InformaticsInstituteofTechnology/DSGP/solar-panel-fault-mapping/.venv/lib/python3.11/site-packages/PIL/Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


43/43 ━━━━━━━━━━━━━━━━━━━━ 218s 5s/step - accuracy: 0.9095 - auc: 0.9505 - loss: 0.2435 - precision: 0.9435 - recall: 0.9088 - val_accuracy: 0.9892 - val_auc: 0.9996 - val_loss: 0.0974 - val_precision: 0.9925 - val_recall: 0.9900
Epoch 2/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 198s 5s/step - accuracy: 0.9963 - auc: 0.9998 - loss: 0.0262 - precision: 0.9951 - recall: 0.9991 - val_accuracy: 0.9923 - val_auc: 0.9999 - val_loss: 0.0419 - val_precision: 0.9877 - val_recall: 1.0000
Epoch 3/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 215s 5s/step - accuracy: 0.9994 - auc: 1.0000 - loss: 0.0126 - precision: 0.9991 - recall: 1.0000 - val_accuracy: 0.9954 - val_auc: 0.9999 - val_loss: 0.0206 - val_precision: 0.9975 - val_recall: 0.9950
Epoch 4/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 210s 5s/step - accuracy: 0.9990 - auc: 1.0000 - loss: 0.0096 - precision: 0.9995 - recall: 0.9990 - val_accuracy: 0.9954 - val_auc: 0.9999 - val_loss: 0.0134 - val_precision: 0.9950 - val_recall: 0.9975
Epoch 5/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 224s 5s/s

...


Saved: /Users/seyedrumaiz/dl_mlflow_outputs/all_models_summary.csv
            model  accuracy  precision  ...  roc_auc    pr_auc  log_loss
0     DenseNet201  0.998462   0.997506  ...  0.99997  0.999981  0.011241
1        ResNet50  0.824615   0.993103  ...  0.99167  0.994286  0.363587
2  EfficientNetB0  0.786154   0.752418  ...  0.90176  0.935804  0.488644

[3 rows x 8 columns]


,model,accuracy,precision,recall,f1,log_loss,roc_auc,pr_auc,ks,ks_threshold,brier,roc_auc_curve_auc,pr_curve_ap
0,DenseNet201,0.998462,0.997506,1.0000,0.998752,0.011241,0.99997,0.999981,0.9960,0.817528,0.002366,0.99997,0.999981
1,ResNet50,0.824615,0.993103,0.7200,0.834783,0.363587,0.99167,0.994286,0.9245,0.293500,0.117243,0.99167,0.994286
2,EfficientNetB0,0.786154,0.752418,0.9725,0.848419,0.488644,0.90176,0.935804,0.6475,0.618209,0.158520,0.90176,0.935804


<Figure size 600x500 with 0 Axes>

<Figure size 600x500 with 0 Axes>

<Figure size 600x500 with 0 Axes>

## Model Comparison Discussion

DenseNet201 significantly outperformed the other backbones across all evaluation metrics, achieving near-perfect ROC-AUC, PR-AUC, and F1-score.

ResNet achieved high precision (0.993) but lower recall, indicating an increased amount of fault negatives. Although ROC-AUC remained high, overall classification balance was weaker.

EfficientNetB0 was the weakest, due its decrease in metrics compared to the other two models.

Based on comprehensive evaluation, DenseNet201 is selected as the final model.